# Exploring the review flow

A hands-on walk through what `agentic-review review` does, one stage at a time, so you can see and change each
input. It uses the **real** code in `src/agentic_review/`, not a copy.

| § | Stage | Calls a model? |
|---|---|---|
| 0 | Settings and a server check | no |
| 1 | The demo repository | no |
| 2 | Change set: what git says changed | no |
| 3 | Assembly: what the model is sent | no |
| 4 | The `emit_finding` tool schema | no |
| 5 | Validation playground | no |
| 6 | One raw model call | **yes** |
| 7 | The review step (graph, observations, outcome) | **yes** |
| 8 | The whole pipeline, terminal and JSON | **yes** |
| 9 | Score against the planted defects | no (uses §8) |
| 10 | Can this model call tools at all? | **yes** |
| 11 | Compare models (a mini S-1) | **yes** |
| 12 | Prompt experiments | **yes** |
| 13 | Offline, with the scripted stand-in model | no |

**Before you start**

- Run this with the project's environment: select the `.venv` kernel of this project (`uv sync` installs `ipykernel`).
- A local OpenAI-compatible server (Ollama by default) with a model that **supports tool calling**. From the
  explorations so far: `qwen2.5-coder:0.5b` does not manage it; try `qwen2.5-coder:7b`, `qwen3:8b` or `llama3.1:8b`.
- Edit **§0** to change the model, the server or the demo location. Everything below reads from there.

This is a scratch space: nothing here changes the tool's behaviour, and nothing is written outside the demo repository.

## 0 · Settings and a server check

In [1]:
import json
import os
import subprocess
import sys
import time
import urllib.request
from dataclasses import replace
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT))  # so `tests.support` and `scripts` import

# ---- change these ---------------------------------------------------------------------------------------------------
BASE_URL = "https://api.groq.com/openai/v1"  # NVIDIA NIM (SRS \u00a72.3, same default as AgentSettings.base_url)
MODEL_NAME = "openai/gpt-oss-120b"                         # must support tool calling
TIMEOUT_SECONDS = 300
DEMO = PROJECT.parent / "review-demo"            # built by scripts/make_review_demo.py; keep it OUTSIDE the project
# ---------------------------------------------------------------------------------------------------------------------


def read_env_file(path: Path) -> dict[str, str]:
    values = {}
    if path.is_file():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, _, value = line.partition("=")
                values[key.strip()] = value.strip()
    return values


# A local server ignores the key, but preflight requires one. The value is never printed.
API_KEY = read_env_file(PROJECT / ".env").get("GROQ_API_KEY") or "local"
ENV = {**os.environ, "GROQ_API_KEY": API_KEY}

print("project:", PROJECT)
print("demo:   ", DEMO)
print("server: ", BASE_URL, "| model:", MODEL_NAME)

project: d:\Projects\DEPI-industry-project\Pull-request-Reviewer
demo:    d:\Projects\DEPI-industry-project\review-demo
server:  https://api.groq.com/openai/v1 | model: openai/gpt-oss-120b


#### Ollama only part

In [2]:
# def get_json(url: str, payload: dict | None = None) -> dict:
#     data = json.dumps(payload).encode() if payload is not None else None
#     request = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
#     with urllib.request.urlopen(request, timeout=10) as response:
#         return json.load(response)


# served = sorted(m["id"] for m in get_json(f"{BASE_URL}/models")["data"])
# print("models served:", served)
# print("configured model served:", MODEL_NAME in served)

# # Ollama only: what the server says the model can do. "tools" must be listed (necessary, not sufficient).
# if ":11434" in BASE_URL:
#     for name in served:
#         capabilities = get_json(BASE_URL.replace("/v1", "/api/show"), {"model": name}).get("capabilities", [])
#         print(f"  {name:28} {capabilities}")

## 1 · The demo repository

A clean commit, then uncommitted edits with six planted defects, plus an excluded build file, a Markdown file, a
deleted file and an untracked new file. Set `REBUILD = True` to start fresh.

In [3]:
REBUILD = False

if REBUILD or not DEMO.exists():
    args = [sys.executable, str(PROJECT / "scripts" / "make_review_demo.py"), "--dest", str(DEMO)]
    print(subprocess.run(args + (["--force"] if DEMO.exists() else []), capture_output=True, text=True).stdout)

print(subprocess.run(["git", "status", "--short"], cwd=DEMO, capture_output=True, text=True).stdout)

 M README.md
 M dist/bundle.js
 M src/format.js
 D src/legacy.ts
 M src/search.ts
 M src/users.ts
?? PLANTED_DEFECTS.md
?? src/retry.ts



In [4]:
import importlib.util

spec = importlib.util.spec_from_file_location("make_review_demo", PROJECT / "scripts" / "make_review_demo.py")
demo_script = importlib.util.module_from_spec(spec)
spec.loader.exec_module(demo_script)

PLANTED = demo_script.PLANTED  # (file, lines, category, expected severity, description)
for file, lines, category, severity, text in PLANTED:
    print(f"{file}:{lines:6} [{severity}, {category}] {text}")

src/users.ts:6      [important or blocking, error-handling] `return fetchUser(id)` without `await`: a rejected promise escapes the try/catch, so the function rejects instead of returning null.
src/users.ts:15-16  [blocking, correctness] `i <= users.length` reads one element past the end (`users[users.length].age` throws).
src/users.ts:13-19  [important, correctness] The empty-array guard was removed: `averageAge([])` now returns NaN (0 / 0).
src/search.ts:4      [blocking, security] The parameterised query became string concatenation: SQL injection through `name`.
src/retry.ts:5-7    [important, error-handling] New file: every error is swallowed and the caller gets `undefined` with no cause, even for non-retryable errors.
src/format.js:2      [nit, maintainability] `unused` is computed and never read.


## 2 · Change set — what git says changed

`resolve()` is the only place git runs. Try editing a file in the demo repository, adding an `exclude` pattern, or
creating a file under `node_modules/`, then re-run.

In [5]:
from agentic_review.core.changeset.model import ScopeRequest
from agentic_review.core.changeset.resolve import resolve
from agentic_review.core.changeset.summary import measure_new_files, summarise

EXTRA_EXCLUDES: list[str] = []          # e.g. ["src/*.js"]

change_set = resolve(DEMO, ScopeRequest(kind="working-tree"), EXTRA_EXCLUDES, was_defaulted=True)
summary = summarise(change_set)
line_counts = measure_new_files(change_set)



In [6]:
for file in change_set.files:
    print(f"file_path: {file.path}")
    print(file.hunks)
    print("===============")
    # for hunk in file.hunks:
    #     print(f"  {hunk.start_line}:{hunk.end_line} [{hunk.category}] {hunk.description}")

file_path: PLANTED_DEFECTS.md
()
file_path: README.md
()
file_path: src/format.js
(Hunk(old_start=1, old_count=3, new_start=1, new_count=4, header='@@ -1,3 +1,4 @@', text=' export function formatPrice(cents) {\n+  const unused = cents * 2;\n   return `$${(cents / 100).toFixed(2)}`;\n }\n'),)
file_path: src/legacy.ts
()
file_path: src/retry.ts
(Hunk(old_start=0, old_count=0, new_start=1, new_count=10, header='@@ -0,0 +1,10 @@', text='+export async function withRetry<T>(operation: () => Promise<T>, attempts = 3): Promise<T | undefined> {\n+  for (let attempt = 1; attempt <= attempts; attempt++) {\n+    try {\n+      return await operation();\n+    } catch {\n+      // try again\n+    }\n+  }\n+  return undefined;\n+}\n'),)
file_path: src/search.ts
(Hunk(old_start=1, old_count=5, new_start=1, new_count=5, header='@@ -1,5 +1,5 @@', text=' import { db } from "./db";\n \n export async function findUsersByName(name: string) {\n-  return db.query("SELECT id, name FROM users WHERE name = $1", [

In [7]:


print(change_set.scope.description)
print(f"\n{'path':24} {'change':9} {'lang':6} {'binary':6} {'hunks':5} {'+/-':7} status")
for f in summary.files:
    status = "reviewed" if f.reviewed else f"not reviewed: {f.not_reviewed_reason}"
    print(f"{f.path:24} {f.change_type:9} {f.language:6} {str(f.is_binary):6} {f.hunk_count:<5} "
          f"+{f.added_lines}/-{f.removed_lines:<4} {status}")
print("\nexcluded:", [(e.path, e.rule) for e in change_set.excluded])
print("line counts (reviewed version):", dict(line_counts))

uncommitted changes in the working tree (default) · 7 files

path                     change    lang   binary hunks +/-     status
PLANTED_DEFECTS.md       added     other  False  0     +20/-0    not reviewed: not-typescript-or-javascript
README.md                modified  other  False  0     +1/-1    not reviewed: not-typescript-or-javascript
src/format.js            modified  js     False  1     +1/-0    reviewed
src/legacy.ts            deleted   ts     False  0     +0/-1    not reviewed: deleted
src/retry.ts             added     ts     False  1     +10/-0    reviewed
src/search.ts            modified  ts     False  1     +1/-1    reviewed
src/users.ts             modified  ts     False  2     +2/-5    reviewed

excluded: [('dist/bundle.js', 'default:build-output')]
line counts (reviewed version): {'src/format.js': 4, 'src/retry.ts': 10, 'src/search.ts': 5, 'src/users.ts': 19}


In [8]:
# One file in detail: hunks exactly as git produced them, and the changed line ranges in new-file coordinates.
FILE = "src/users.ts"

changed = next(f for f in change_set.files if f.path == FILE)
print("changed line ranges:", changed.changed_line_ranges)
for index, hunk in enumerate(changed.hunks):
    print(f"\n[hunk {index}] {hunk.header}")
    print(hunk.text, end="")

changed line ranges: ((6, 6), (15, 15))

[hunk 0] @@ -3,7 +3,7 @@ import { logger } from "./logger";
 
 export async function loadUser(id: string): Promise<User | null> {
   try {
-    return await fetchUser(id);
+    return fetchUser(id);
   } catch (error) {
     logger.warn("user lookup failed", error);
     return null;

[hunk 1] @@ -11,11 +11,8 @@ export async function loadUser(id: string): Promise<User | null> {
 }
 
 export function averageAge(users: User[]): number {
-  if (users.length === 0) {
-    return 0;
-  }
   let total = 0;
-  for (let i = 0; i < users.length; i++) {
+  for (let i = 0; i <= users.length; i++) {
     total += users[i].age;
   }
   return total / users.length;


## 3 · Assembly — exactly what the model is sent

Two messages: the system instructions, then the "no checks / no context" statement followed by every reviewable
hunk verbatim. The token count is an estimate (characters ÷ 3).

In [9]:
from agentic_review.core.assembly.prompt import assemble, prompt_text, user_message
from agentic_review.core.config.settings import AgentSettings, BudgetSettings

context = assemble(change_set, line_counts, BudgetSettings())

print("estimated tokens:", context.estimated_tokens)
print("material the findings are checked against:", dict(context.material.reviewed_files))
print("\n================ SYSTEM ================\n")
print(context.instructions)
print("================ USER ================\n")
print(user_message(context.material_statement, context.diff_section))

estimated tokens: 2107
material the findings are checked against: {'src/format.js': ReviewedFile(line_count=4, hunk_count=1), 'src/retry.ts': ReviewedFile(line_count=10, hunk_count=1), 'src/search.ts': ReviewedFile(line_count=5, hunk_count=1), 'src/users.ts': ReviewedFile(line_count=19, hunk_count=2)}

================ SYSTEM ================

You are an experienced TypeScript/JavaScript reviewer. You are reviewing one change. You are given the complete diff of every file under review, and nothing else: no test results, no other files, no history.

# How to report

You get exactly one turn: there is no later turn in which to report more problems. If you find multiple problems, invoke `emit_finding` multiple times **within this same turn** — one call per problem, all issued together alongside each other, not one now while you wait for a result before issuing the next. For example, three problems means three `emit_finding` calls emitted together in this one turn, never spread across turn

## 4 · The `emit_finding` tool — the only way a finding gets in

In [10]:
from agentic_review.agents.review_port import emit_finding_tool

print(json.dumps(emit_finding_tool(), indent=2))

{
  "type": "function",
  "function": {
    "name": "emit_finding",
    "description": "Record ONE review finding: a single specific problem, at one location in one reviewed file. Call this once per problem, as soon as you find it. This call is the only way a finding is recorded \u2014 prose in your reply is discarded unread. Each call is validated against the change under review: the file must be one of the reviewed files, the line range must exist in its new version, and the evidence must point at something this review actually has. A rejected call records nothing.",
    "parameters": {
      "additionalProperties": false,
      "description": "One review finding: a single specific problem, at one location, in one file of the change under review.",
      "properties": {
        "file": {
          "description": "Repository-relative path of the reviewed file the problem is in, exactly as the diff shows it.",
          "minLength": 1,
          "type": "string"
        },
        "sta

## 5 · Validation playground — no model needed

Every tool call goes through `validate_call` against **this run's** material: shape first, then references (the file
was reviewed, the lines exist, the cited hunk exists, no checks/context/rules are cited). The rejection reason is
what the model would be told. Edit the arguments and re-run.

In [11]:
from agentic_review.core.findings.validation import Accepted, validate_call

candidates = {
    "valid, cites a hunk": {
        "file": "src/search.ts", "startLine": 4, "endLine": 4, "severity": "blocking", "category": "security",
        "rationale": "User input is concatenated into SQL.", "evidence": {"kind": "diff", "file": "src/search.ts", "hunkIndex": 0},
    },
    "valid, unverified": {
        "file": "src/retry.ts", "startLine": 5, "endLine": 7, "severity": "important", "category": "error-handling",
        "rationale": "Errors are swallowed.", "evidence": {"kind": "unverified", "reason": "intent not visible in the diff"},
    },
    "file not reviewed": {
        "file": "README.md", "startLine": 1, "endLine": 1, "severity": "nit", "category": "maintainability",
        "rationale": "Typo.", "evidence": {"kind": "unverified", "reason": "n/a"},
    },
    "line past the end": {
        "file": "src/search.ts", "startLine": 4, "endLine": 99, "severity": "blocking", "category": "security",
        "rationale": "x", "evidence": {"kind": "diff", "file": "src/search.ts"},
    },
    "bad severity, extra field": {
        "file": "src/search.ts", "startLine": 4, "endLine": 4, "severity": "critical", "category": "security",
        "rationale": "x", "evidence": {"kind": "diff", "file": "src/search.ts"}, "confidence": 0.9,
    },
    "cites test results that never ran": {
        "file": "src/users.ts", "startLine": 15, "endLine": 16, "severity": "blocking", "category": "correctness",
        "rationale": "Off by one.", "evidence": {"kind": "check", "checkId": "tests"},
    },
    "what the 0.5b model produced": {"cents": 1234567890},
}

for label, args in candidates.items():
    result = validate_call("emit_finding", args, context.material)
    verdict = "ACCEPTED" if isinstance(result, Accepted) else f"REJECTED  {result.reason}"
    print(f"{label:34} {verdict}")

valid, cites a hunk                ACCEPTED
valid, unverified                  ACCEPTED
file not reviewed                  REJECTED  file: 'README.md' was not part of this review; reviewed files: src/format.js, src/retry.ts, src/search.ts, src/users.ts
line past the end                  REJECTED  endLine: 99 is past the end of 'src/search.ts' (5 lines)
bad severity, extra field          REJECTED  severity: Input should be 'blocking', 'important' or 'nit'; confidence: Extra inputs are not permitted
cites test results that never ran  REJECTED  evidence.kind: no checks were run in this review; cite the diff or mark the finding unverified
what the 0.5b model produced       REJECTED  file: Field required; startLine: Field required; endLine: Field required; severity: Field required; category: Field required; rationale: Field required; evidence: Field required; cents: Extra inputs are not permitted


## 6 · One raw model call

The same two messages and the same tool the review sends, but you see the reply **before** the system interprets it.
What to look for:

- `tool_calls` non-empty → the server parsed real tool calls. Good.
- `tool_calls` empty and `content` holds JSON → the model *imitated* a call in text. It will be discarded (prose is
  never parsed for findings) and the run records `no-tool-call`.
- `invalid_tool_calls` non-empty → a call whose arguments were not valid JSON.

In [12]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


def chat_model(model: str = MODEL_NAME, temperature: float = 0.0) -> ChatOpenAI:
    return ChatOpenAI(base_url=BASE_URL, model=model, api_key=API_KEY, temperature=temperature,
                      timeout=TIMEOUT_SECONDS, max_retries=0)


def raw_reply(model: str = MODEL_NAME, instructions: str | None = None):
    bound = chat_model(model).bind_tools([emit_finding_tool()])
    messages = [
        SystemMessage(content=instructions or context.instructions),
        HumanMessage(content=user_message(context.material_statement, context.diff_section)),
    ]
    started = time.perf_counter()
    reply = bound.invoke(messages)
    return reply, time.perf_counter() - started


reply, seconds = raw_reply()
print(f"latency: {seconds:.1f}s   finish_reason: {reply.response_metadata.get('finish_reason')}   "
      f"tokens: {reply.usage_metadata}")
print("tool_calls:", len(reply.tool_calls), "| invalid_tool_calls:", len(reply.invalid_tool_calls))
for call in reply.tool_calls:
    print(" ", call["name"], json.dumps(call["args"]))
print("\ncontent:\n", reply.content[:3000])



latency: 4.4s   finish_reason: tool_calls   tokens: {'input_tokens': 2625, 'output_tokens': 1884, 'total_tokens': 4509, 'input_token_details': {}, 'output_token_details': {'reasoning': 1768}}
tool_calls: 1 | invalid_tool_calls: 0
  emit_finding {"category": "maintainability", "endLine": 2, "evidence": {"file": "src/format.js", "hunkIndex": 0, "kind": "diff"}, "file": "src/format.js", "rationale": "The variable `unused` is calculated but never used, adding dead code and potential confusion.", "severity": "nit", "startLine": 2}

content:
 


In [13]:
from IPython.display import display, Markdown, HTML, JSON
display(JSON(reply.to_json()))

<IPython.core.display.JSON object>

## 7 · The two graphs — the pipeline, and CF-6 inside it

Since [ADR-0009](../Docs/adr/0009-pipeline-as-declared-graph.md) the **order of operations is itself a graph**: one node per phase of SRS §2.1, including the deterministic ones and the three whose capability is not built yet (`parse_and_context` · CF-2, `execute_checks` · CF-3, `load_patterns` · CF-8 are pass-throughs today).

The pipeline graph is **acyclic**. CF-6's graph — the review node — is the only place a cycle may occur, and that cycle is the single repair turn. Neither cell calls the provider, so both are free to run without a key.

In [ ]:
from agentic_review.agents.review_reporting_agent import build_review_graph
from agentic_review.agents.pipeline_agent import build_pipeline_graph

print("PIPELINE — CF-1 … CF-9, one node per phase")
print(build_pipeline_graph().get_graph().draw_ascii())

print("\nCF-6 — the review step, the only graph with a cycle")
compiled = build_review_graph(chat_model().bind_tools([emit_finding_tool()]))
print(compiled.get_graph().draw_ascii())

## 8 · The review step — graph, observations, outcome

`build_review_port` binds the tool and runs the LangGraph review step. `last_transcript` shows every message,
including the structured observation each tool call received (`ok` / `error` with the reason).

In [15]:
from agentic_review.agents.review_port import build_review_port
from agentic_review.core.config.credentials import Credential

settings = AgentSettings(base_url=BASE_URL, model=MODEL_NAME, request_timeout_seconds=TIMEOUT_SECONDS)
credential = Credential(API_KEY, source="environment", location="GROQ_API_KEY")

port = build_review_port(settings, credential, chat_model=chat_model())
started = time.perf_counter()
outcome = port.review(context)
print(f"{time.perf_counter() - started:.1f}s  completion={outcome.completion_status}  "
      f"calls={outcome.tool_calls_total}  findings={len(outcome.findings)}  tokens={outcome.tokens_in}/{outcome.tokens_out}")
for d in outcome.degradations:
    print(f"degradation [{d.source}] {d.code}: {d.detail}")

print("\n--- transcript ---")
for message in port.last_transcript:
    body = message.content if isinstance(message.content, str) else json.dumps(message.content)
    extra = f"  tool_calls={[c['name'] for c in message.tool_calls]}" if getattr(message, "tool_calls", None) else ""
    print(f"[{message.type}]{extra} {body[:400]}")

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01j8mvxhckfjj941zqmnmcatar` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6704, Requested 3097. Please try again in 13.5075s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
for finding in outcome.findings:
    print(json.dumps(finding.model_dump(mode="json", by_alias=True), indent=2))

{
  "file": "src/format.js",
  "startLine": 2,
  "endLine": 2,
  "severity": "nit",
  "category": "maintainability",
  "rationale": "The variable `unused` is calculated but never used, adding dead code that serves no purpose.",
  "evidence": {
    "kind": "diff",
    "file": "src/format.js",
    "hunkIndex": 0
  },
  "id": "f-1",
  "schemaVersion": "1.0",
  "turnIndex": 0,
  "fingerprint": "a8e97ab85c25"
}


## 9 · The whole pipeline — what the CLI runs

`run_review` does preflight → change set → assembly → review → output and times each stage. `port_factory` swaps
in the model chosen in §0 without touching `settings.py` or the demo's `.agentic-review.yml`.

In [16]:
from rich.console import Console

from agentic_review.cli.exit_codes import exit_code_for
from agentic_review.core.output.json_out import to_json
from agentic_review.core.output.terminal import render_terminal
from agentic_review.agents.pipeline_agent import ReviewRequest, Reviewed, run_review


def notebook_port(settings_from_config, key):
    chosen = settings_from_config.model_copy(
        update={"base_url": BASE_URL, "model": MODEL_NAME, "request_timeout_seconds": TIMEOUT_SECONDS}
    )
    return build_review_port(chosen, key, chat_model=chat_model())


result = run_review(ReviewRequest(cwd=DEMO), env=ENV, port_factory=notebook_port)
if not isinstance(result, Reviewed):
    print("PREFLIGHT FAILED:\n" + result.message)
else:
    report = result.report
    render_terminal(report, Console(width=140, force_jupyter=False, force_terminal=False, no_color=True))
    print("\nexit code would be:", exit_code_for(report.verdict))
    print("stage durations (ms):", report.run.stage_durations_ms.model_dump(by_alias=True))

Scope: uncommitted changes in the working tree (default) · 7 files
Reviewed (4):
  src/format.js
  src/retry.ts
  src/search.ts
  src/users.ts
Listed, not reviewed (3):
  PLANTED_DEFECTS.md (not TypeScript or JavaScript)
  README.md (not TypeScript or JavaScript)
  src/legacy.ts (deleted)
Excluded (1):
  dist/bundle.js (default:build-output)

No checks were run in this build.
No related context was gathered in this build.

src/format.js:2  nit  maintainability
  The variable `unused` is calculated but never used, adding dead code that serves no purpose.
  evidence: diff hunk 0

src/retry.ts:5-7  important  error-handling
  The catch block swallows all errors and the function returns undefined on failure, losing error information and making failures indistinguishable from a successful undefined result.
  evidence: diff hunk 0

Findings: 0 blocking / 1 important / 1 nit
Degradation [CF-6] incomplete-file-coverage: No emit_finding call, accepted or rejected, ever named: src/search.ts, src

In [17]:
# The JSON report the same run produced (no --json flag yet on the CLI; this is the same object).
print(to_json(report))

{
  "schemaVersion": "1.0",
  "run": {
    "startedAt": "2026-09-17T21:09:04Z",
    "durationMs": 7078,
    "stageDurationsMs": {
      "preflight": 32,
      "changeSet": 187,
      "assembly": 0,
      "review": 6859,
      "output": 0
    },
    "requestedDepth": null,
    "achievedDepth": null,
    "toolCallCeiling": 25,
    "investigationToolsEnabled": false,
    "toolCallsTotal": 2,
    "toolCallsEmission": 2,
    "toolCallsInvestigation": 0,
    "repairTurns": 1,
    "completionStatus": "emitted",
    "emitPath": "tool-call",
    "model": "openai/gpt-oss-120b",
    "provider": "api.groq.com",
    "sampling": {
      "temperature": 0.0,
      "seed": null
    },
    "tokensIn": 5462,
    "tokensOut": 2654,
    "budgetLimit": 20000,
    "budgetHit": false,
    "schemaVersion": "1.0",
    "toolVersion": "0.1.0"
  },
  "changeSummary": {
    "scope": {
      "kind": "working-tree",
      "baseRef": null,
      "headRef": null,
      "wasDefaulted": true,
      "description": "uncomm

## 10 · Score against the planted defects

A rough match: a finding counts as catching a planted defect when it is in the same file and its line range overlaps.
This is a feel, not a measurement (the real one is feature 003's regression harness).

In [18]:
def line_span(text: str) -> tuple[int, int]:
    start, _, end = text.partition("-")
    return int(start), int(end or start)


def overlaps(a: tuple[int, int], b: tuple[int, int]) -> bool:
    return a[0] <= b[1] and b[0] <= a[1]


matched_findings = set()
for file, lines, category, severity, text in PLANTED:
    hits = [f for f in report.findings if f.file == file and overlaps((f.start_line, f.end_line), line_span(lines))]
    matched_findings.update(f.id for f in hits)
    mark = "CAUGHT " if hits else "missed "
    detail = ", ".join(f"{f.id} {f.severity}/{f.category}" for f in hits)
    print(f"{mark} {file}:{lines:6} {text[:70]}  {detail}")

caught = sum(
    1 for file, lines, *_ in PLANTED
    if any(f.file == file and overlaps((f.start_line, f.end_line), line_span(lines)) for f in report.findings)
)
extra = [f for f in report.findings if f.id not in matched_findings]
print(f"\ncaught {caught}/{len(PLANTED)} planted defects; {len(extra)} finding(s) matched nothing planted")
for f in extra:
    print(f"  {f.id} {f.file}:{f.start_line}-{f.end_line} {f.severity}/{f.category}: {f.rationale}")

missed  src/users.ts:6      `return fetchUser(id)` without `await`: a rejected promise escapes the  
missed  src/users.ts:15-16  `i <= users.length` reads one element past the end (`users[users.lengt  
missed  src/users.ts:13-19  The empty-array guard was removed: `averageAge([])` now returns NaN (0  
missed  src/search.ts:4      The parameterised query became string concatenation: SQL injection thr  
CAUGHT  src/retry.ts:5-7    New file: every error is swallowed and the caller gets `undefined` wit  f-2 important/error-handling
CAUGHT  src/format.js:2      `unused` is computed and never read.  f-1 nit/maintainability

caught 2/6 planted defects; 0 finding(s) matched nothing planted


## 11 · Can this model call tools at all?

The easiest possible case: one tool, and a prompt that spells out every argument. If a model fails **this**, it will
not work as a reviewer, whatever the prompt. (`qwen2.5-coder:0.5b` failed 5/5 here.)

In [ ]:
def explicit_tool_call_check(model: str = MODEL, tries: int = 3) -> None:
    bound = chat_model(model).bind_tools([emit_finding_tool()])
    prompt = [
        SystemMessage(content="Report problems only by calling the emit_finding tool."),
        HumanMessage(content=(
            "In src/search.ts line 4, the SQL query concatenates user input: SQL injection. "
            "Call emit_finding with file src/search.ts, startLine 4, endLine 4, severity blocking, category security, "
            'a rationale, and evidence {"kind": "diff", "file": "src/search.ts", "hunkIndex": 0}.'
        )),
    ]
    for attempt in range(1, tries + 1):
        reply = bound.invoke(prompt)
        calls = [(c["name"], validate_call(c["name"], c["args"], context.material)) for c in reply.tool_calls]
        summary = [f"{name}: {'accepted' if isinstance(r, Accepted) else r.reason}" for name, r in calls]
        print(f"{model} try {attempt}: tool_calls={summary or 'none'}  content={reply.content[:100]!r}")


explicit_tool_call_check()

qwen2.5-coder:7b try 1: tool_calls=none  content='{"name": "emit_finding", "arguments": {"file": "src/search.ts", "startLine": 4, "endLine": 4, "sever'
qwen2.5-coder:7b try 2: tool_calls=none  content='{"name": "emit_finding", "arguments": {"file": "src/search.ts", "startLine": 4, "endLine": 4, "sever'
qwen2.5-coder:7b try 3: tool_calls=none  content='{"name": "emit_finding", "arguments": {"file": "src/search.ts", "startLine": 4, "endLine": 4, "sever'


## 12 · Compare models — a mini S-1

Runs the real review step `RUNS` times per model on the demo diff. The questions S-1 asks: does the model call the
tool, are the calls valid, how long does it take? No pass mark — look at the numbers. The full probe is
`scripts/s1_tool_calling_probe.py`.

In [ ]:
MODELS = [MODEL]          # e.g. ["qwen2.5-coder:7b", "qwen3:8b", "llama3.1:8b"]
RUNS = 3

rows = []
for model in MODELS:
    for run in range(1, RUNS + 1):
        port = build_review_port(settings.model_copy(update={"model": model}), credential, chat_model=chat_model(model))
        started = time.perf_counter()
        try:
            o = port.review(context)
            rows.append((model, run, o.completion_status, o.tool_calls_total, len(o.findings),
                         sorted(f.severity for f in o.findings), round(time.perf_counter() - started, 1), ""))
        except Exception as error:  # provider errors are not mapped until User Story 2
            rows.append((model, run, "ERROR", 0, 0, [], round(time.perf_counter() - started, 1), type(error).__name__))

print(f"{'model':24} {'run':3} {'completion':14} {'calls':5} {'valid':5} {'seconds':7} severities")
for model, run, completion, calls, valid, severities, seconds, error in rows:
    print(f"{model:24} {run:<3} {completion:14} {calls:<5} {valid:<5} {seconds:<7} {severities} {error}")

model                    run completion     calls valid seconds severities
qwen2.5-coder:7b         1   no-tool-call   0     0     5.0     [] 
qwen2.5-coder:7b         2   no-tool-call   0     0     5.0     [] 
qwen2.5-coder:7b         3   no-tool-call   0     0     5.2     [] 


## 13 · Prompt experiments

The shipped instructions are deliberately minimal (they get tuned only once feature 003 can measure the effect).
Here you can try variants without touching the code: `replace` swaps the instructions in the assembled context.

In [ ]:
INSTRUCTIONS

'You are an experienced TypeScript/JavaScript reviewer. You are reviewing one change. You are given the complete diff of every file under review, and nothing else: no test results, no other files, no history.\n\n# How to report\n\nReport each problem by calling the `emit_finding` tool, one call per problem. A tool call is the only thing that is recorded. Any prose you write is discarded unread, so never describe a problem in text instead of calling the tool.\n\nWhen you have reported every problem you found, reply without a tool call. If you found nothing worth reporting, reply without a tool call.\n\n# What to report\n\nReport a problem only when all three are true: this change introduced or exposed it, it has a concrete consequence you can name, and a reviewer would ask the author to address it before merging.\n\nLook for, in this order:\n\n1. `correctness` — logic that gives a wrong result: off-by-one, inverted or missing condition, wrong operator, unhandled case, misused API, a gua

In [ ]:
from agentic_review.core.assembly.prompt import INSTRUCTIONS

VARIANT = INSTRUCTIONS + '''
You MUST call emit_finding once for every problem you find. Do not describe problems in text.
Example call arguments:
{"file": "src/a.ts", "startLine": 3, "endLine": 3, "severity": "important", "category": "correctness",
 "rationale": "Why this is wrong.", "evidence": {"kind": "diff", "file": "src/a.ts", "hunkIndex": 0}}
'''

for label, instructions in [("shipped", INSTRUCTIONS), ("variant", VARIANT)]:
    port = build_review_port(settings, credential, chat_model=chat_model())
    o = port.review(replace(context, instructions=instructions))
    print(f"{label:8} completion={o.completion_status:14} calls={o.tool_calls_total} findings={len(o.findings)} "
          f"{[(f.file, f.start_line, f.severity) for f in o.findings]}")

shipped  completion=emitted        calls=1 findings=1 [('src/format.js', 2, 'nit')]
variant  completion=emitted        calls=1 findings=1 [('src/format.js', 2, 'nit')]


## 14 · Offline, with the scripted stand-in model

The test suite never calls a real model. `ScriptedChatModel` replays replies you write, through the **real** graph,
so you can see exactly how the system handles a given reply: accepted, rejected, prose-only, invalid JSON.

In [ ]:
from tests.support.scripted_chat_model import ScriptedChatModel, ai_reply, finding_call, invalid_json_call

script = [
    ai_reply(
        [
            finding_call(file="src/search.ts", startLine=4, endLine=4, severity="blocking", category="security",
                         rationale="SQL injection.", evidence={"kind": "diff", "file": "src/search.ts", "hunkIndex": 0}),
            finding_call(file="src/nope.ts"),                               # rejected: file not reviewed
        ],
        invalid_tool_calls=[invalid_json_call()],                          # rejected: not valid JSON
        content="Some prose that is discarded.",
        usage=(900, 50),
    )
]
stand_in = ScriptedChatModel(script)
port = build_review_port(settings, credential, chat_model=stand_in)
o = port.review(context)

print("model calls:", stand_in.call_count, "| bound kwargs:", stand_in.bound_kwargs)
print("completion:", o.completion_status, "| findings:", [f.id for f in o.findings],
      "| degradations:", [(d.code, d.detail) for d in o.degradations])
for message in port.last_transcript:
    if message.type == "tool":
        print("observation:", message.content)

model calls: 1 | bound kwargs: {}
completion: emitted | findings: ['f-1'] | degradations: [('invalid-model-output', '2 emit_finding call(s) could not be validated; 1 finding(s) were accepted.')]
observation: {"status": "ok", "reason": null, "content": "accepted as f-1"}
observation: {"status": "error", "reason": "file: 'src/nope.ts' was not part of this review; reviewed files: src/format.js, src/retry.ts, src/search.ts, src/users.ts; evidence.file: 'src/nope.ts' was not part of this review", "content": null}
observation: {"status": "error", "reason": "arguments: not valid JSON", "content": null}
